# 06. M0 de referencia

Se conserva el clasificador de polaridad RoBERTuito publicado por pysentimiento. Esta etapa guarda los pesos y el tokenizador de partida. El preprocesador `preprocess_tweet(lang='es')` se usa antes de tokenizar en los ajustes siguientes. M0 corresponde a la condición sin corpus adicional.

La corrida se elige en `corrida.json`. Las evaluaciones reproducidas se guardan en `salidas/<ejecución>/evaluacion`.

Si abres este cuaderno directamente en Colab, ejecuta su primera celda y autoriza Google Drive cuando se solicite. Para recorrer toda la secuencia de una vez, usa `INICIAR_EN_COLAB.ipynb`.

**Modo de revisión docente.** Puedes ejecutar todas las celdas con las opciones predeterminadas. Se verifican los pesos y se leen las predicciones de evaluación guardadas para reconstruir las tablas y gráficas. Los nuevos ajustes requieren activar `ENTRENAR_MODELO`; las nuevas predicciones de evaluación requieren `RECALCULAR_EVALUACION`. `PREGUNTAR_CARGA` permite habilitar las preguntas manuales.

Las salidas visibles al abrir el cuaderno proceden de `corrida_03_epocas`. Los registros de M1 y M2 corresponden a sus ajustes originales de tres épocas. Una nueva ejecución en modo de revisión muestra los registros y las métricas guardadas. Nunca se presenta la lectura de un historial como un nuevo entrenamiento.

In [1]:
# Inicio al abrir este cuaderno directamente en Google Colab.
from pathlib import Path

if not Path("corrida.json").is_file():
    import os
    import subprocess
    import sys

    try:
        from google.colab import drive
    except ImportError as error:
        raise FileNotFoundError("En Jupyter, abre la carpeta que contiene corrida.json.") from error

    proyecto_drive = Path("/content/drive/MyDrive/Tesis_RoBERTuito_M0_M1_M2")
    if not (proyecto_drive / "corrida.json").is_file():
        drive.mount("/content/drive", timeout_ms=180000)
    if not (proyecto_drive / "corrida.json").is_file():
        raise FileNotFoundError("Busca Tesis_RoBERTuito_M0_M1_M2 directamente en Mi unidad.")

    os.chdir(proyecto_drive)
    sys.path.insert(0, str(proyecto_drive))
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(proyecto_drive / "requirements-cuadernos.txt")],
        check=True,
    )
    print("Carpeta de la tesis conectada:", proyecto_drive)

import json
from pathlib import Path

RAIZ = Path.cwd()
CORRIDA = json.loads((RAIZ / "corrida.json").read_text(encoding="utf-8"))["corrida"]
SALIDAS = RAIZ / 'modelos' / CORRIDA

# Primera revisión del modelo y confirmación antes de leer el corpus.
import os
from gestion_modelos import preguntar_modelo, nueva_corrida

MODELO = "M0"
ENTRENAR_MODELO = False  # @param {type:"boolean"}
PREGUNTAR_CARGA = False  # @param {type:"boolean"}
RECALCULAR_EVALUACION = False  # @param {type:"boolean"}

ENTRENAR_MODELO = ENTRENAR_MODELO or os.environ.get("TESIS_PERMITIR_ENTRENAMIENTO") == "1"
RECALCULAR_EVALUACION = RECALCULAR_EVALUACION or ENTRENAR_MODELO
respuesta = os.environ.get("TESIS_RESPUESTA_M0")
if respuesta is None:
    respuesta = "no" if ENTRENAR_MODELO else (None if PREGUNTAR_CARGA else "si")
CARGAR_GUARDADO = preguntar_modelo(
    RAIZ, MODELO, respuesta=respuesta, permitir_preparacion=ENTRENAR_MODELO
)

In [2]:
from entrenamiento import preparar_m0

destino = SALIDAS / MODELO
if CARGAR_GUARDADO:
    print("Se utilizarán los pesos guardados de M0.")
else:
    if not ENTRENAR_MODELO:
        raise RuntimeError("Activa ENTRENAR_MODELO para solicitar un ajuste nuevo. La revisión utiliza los pesos incluidos.")
    if destino.exists():
        CORRIDA = nueva_corrida(RAIZ, MODELO)
        SALIDAS = RAIZ / "modelos" / CORRIDA
        destino = SALIDAS / MODELO
    preparar_m0(destino)

M0 guardado: modelos\corrida_03_epocas\M0


## Evaluación de M0

Esta sección se puede ejecutar desde aquí con los pesos guardados. Usa las **454 publicaciones**
de la partición boliviana marcadas como `validacion` en el archivo del split de 1.809/454.
Para la corrida experimental sirven como conjunto de evaluación.

Cada publicación aporta un voto. Se usa su `texto_procesado`, se aplica
`preprocess_tweet(lang='es')` y se tokeniza en tramos de hasta 128 tokens con 32 de
solapamiento. Se promedian las probabilidades de los tramos y se toma la clase de mayor
probabilidad. La referencia es `etiqueta_humana`.

In [3]:
from rutas import carpeta_salidas

import json
from pathlib import Path

import pandas as pd

from evaluar import evaluar_modelo, mostrar_cuadro, resumir

CORRIDA_EVALUADA = json.loads(Path("corrida.json").read_text(encoding="utf-8"))["corrida"]
predicciones = evaluar_modelo(Path.cwd(), CORRIDA_EVALUADA, "M0",
    recalcular=RECALCULAR_EVALUACION,
    directorio_evaluacion=carpeta_salidas(Path.cwd()) / "evaluacion",
)

La matriz de confusión (véase Cuadro 1) cuenta publicaciones por etiqueta humana (fila)
y predicción (columna). La diagonal muestra los aciertos. El Cuadro 2 muestra la
exactitud (*accuracy*), que divide los aciertos entre las 454 publicaciones, y el F1
macro, que promedia el F1 de las tres clases con el mismo peso para cada una.

In [4]:
metricas, matriz = resumir(predicciones)
mostrar_cuadro("Cuadro 1. Matriz de confusión de M0", matriz)
mostrar_cuadro("Cuadro 2. Exactitud y F1 macro de M0", pd.DataFrame([metricas], index=["M0"]))
print(f"Aciertos: {matriz.to_numpy().trace()} de {len(predicciones)}")

Predicción,negativa,neutral,positiva
Etiqueta humana,,,
negativa,280,39,7
neutral,18,73,3
positiva,9,15,10


,Exactitud,F1 macro
M0,79.96%,63.86%


Aciertos: 363 de 454
